In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [0]:
spark=SparkSession.builder.appName("Amazon").getOrCreate()

In [0]:
df=spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema", False) \
    .option("mode","PERMISSIVE")\
    .load("/Volumes/workspace/default/amazon_data/test.csv")


In [0]:
df.show()

+----------+--------------------+--------------------+--------------------+--------------------+
|PRODUCT_ID|               TITLE|       BULLET_POINTS|         DESCRIPTION|     PRODUCT_TYPE_ID|
+----------+--------------------+--------------------+--------------------+--------------------+
|    604373|Manuel d'Héliogra...|                NULL|                NULL|                6142|
|   1729783|"DCGARING Microfi...|"[QUALITY GUARANT...|  all season choose.|SATISFACTION GUAR...|
|   1871949|I-Match Auto Part...|[Front License Pl...|Replacement for T...|                7540|
|   1107571|PinMart Gold Plat...|[Available as a s...|Each pin comes wi...|Our Excellence in...|
|    624253|Visual Mathematic...|                NULL|                NULL|                6318|
|   2782548|Evershine Shoppee...|[Kindly Refer The...|Description: Mate...|               11205|
|   1605901|Vasque Men's Velo...|[The velocity all...|                NULL|                3234|
|    938007|Pokemon M-037 Kyu.

In [0]:
print(df.columns)

['PRODUCT_ID', 'TITLE', 'BULLET_POINTS', 'DESCRIPTION', 'PRODUCT_TYPE_ID']


In [0]:
print("Total Rows:", df.count())

Total Rows: 734736


In [0]:
df.printSchema()

root
 |-- PRODUCT_ID: string (nullable = true)
 |-- TITLE: string (nullable = true)
 |-- BULLET_POINTS: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- PRODUCT_TYPE_ID: string (nullable = true)



In [0]:
print("Number of Columns:", len(df.columns))

Number of Columns: 5


In [0]:
from pyspark.sql.types import StructType,StructField
from pyspark.sql.types import IntegerType,StringType

schema = StructType([
    StructField("PRODUCT_ID", IntegerType(), True),
    StructField("TITLE", StringType(), True),
    StructField("BULLET_POINTS", StringType(), True),
    StructField("DESCRIPTION", StringType(), True),
    StructField("PRODUCT_TYPE_ID", IntegerType(), True),
    StructField("_corrupt_record", StringType(), True)
])

In [0]:
df.printSchema()

root
 |-- PRODUCT_ID: string (nullable = true)
 |-- TITLE: string (nullable = true)
 |-- BULLET_POINTS: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- PRODUCT_TYPE_ID: string (nullable = true)



In [0]:
df = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .schema(schema) \
    .csv("/Volumes/workspace/default/amazon_data/test.csv")

In [0]:
df.printSchema()

root
 |-- PRODUCT_ID: integer (nullable = true)
 |-- TITLE: string (nullable = true)
 |-- BULLET_POINTS: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- PRODUCT_TYPE_ID: integer (nullable = true)
 |-- _corrupt_record: string (nullable = true)



In [0]:
df.select(
    df.PRODUCT_ID.alias("Product_ID"),
    df.TITLE.alias("Product_Title"),
    df.PRODUCT_TYPE_ID.alias("Category_ID")
).show()

+----------+--------------------+-----------+
|Product_ID|       Product_Title|Category_ID|
+----------+--------------------+-----------+
|    604373|Manuel d'Héliogra...|       6142|
|   1729783|"DCGARING Microfi...|       NULL|
|   1871949|I-Match Auto Part...|       7540|
|   1107571|PinMart Gold Plat...|       NULL|
|    624253|Visual Mathematic...|       6318|
|   2782548|Evershine Shoppee...|      11205|
|   1605901|Vasque Men's Velo...|       3234|
|    938007|Pokemon M-037 Kyu...|          0|
|    708128|Ganz normal verrü...|      10512|
|   1609597|Lug Women's Puddl...|       2640|
|    500777|Notebook: Japanes...|      12415|
|   2736605|SHASAK Sanganer H...|       2911|
|   2217122|Mon journal intim...|          1|
|    914243|Carnation Home Fa...|       1348|
|    408556|Paz Interior Livr...|       6337|
|   2896817|RAIBA Realme 9 Pr...|       2210|
|    172273|Ice and Mixed Cli...|        160|
|      1718|House of Glass: T...|       6111|
|    308161|The Works of Samu...| 

In [0]:
df.filter(df.PRODUCT_TYPE_ID > 5000).show()

+----------+--------------------+--------------------+--------------------+---------------+---------------+
|PRODUCT_ID|               TITLE|       BULLET_POINTS|         DESCRIPTION|PRODUCT_TYPE_ID|_corrupt_record|
+----------+--------------------+--------------------+--------------------+---------------+---------------+
|    604373|Manuel d'Héliogra...|                NULL|                NULL|           6142|           NULL|
|   1871949|I-Match Auto Part...|[Front License Pl...|Replacement for T...|           7540|           NULL|
|    624253|Visual Mathematic...|                NULL|                NULL|           6318|           NULL|
|   2782548|Evershine Shoppee...|[Kindly Refer The...|Description: Mate...|          11205|           NULL|
|    708128|Ganz normal verrü...|                NULL|                NULL|          10512|           NULL|
|    500777|Notebook: Japanes...|                NULL|                NULL|          12415|           NULL|
|    408556|Paz Interior Liv

In [0]:
from pyspark.sql.functions import lit
df = df.withColumn("Data_Source", lit("Amazon"))
df.show(5)

+----------+--------------------+--------------------+--------------------+---------------+--------------------+-----------+
|PRODUCT_ID|               TITLE|       BULLET_POINTS|         DESCRIPTION|PRODUCT_TYPE_ID|     _corrupt_record|Data_Source|
+----------+--------------------+--------------------+--------------------+---------------+--------------------+-----------+
|    604373|Manuel d'Héliogra...|                NULL|                NULL|           6142|                NULL|     Amazon|
|   1729783|"DCGARING Microfi...|"[QUALITY GUARANT...|  all season choose.|           NULL|1729783,"DCGARING...|     Amazon|
|   1871949|I-Match Auto Part...|[Front License Pl...|Replacement for T...|           7540|                NULL|     Amazon|
|   1107571|PinMart Gold Plat...|[Available as a s...|Each pin comes wi...|           NULL|1107571,PinMart G...|     Amazon|
|    624253|Visual Mathematic...|                NULL|                NULL|           6318|                NULL|     Amazon|


In [0]:
from pyspark.sql.functions import length
df = df.withColumn("TITLE_LENGTH", length(col("TITLE")))
df.select("TITLE", "TITLE_LENGTH").show(10, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+
|TITLE                                                                                                                                                                                       |TITLE_LENGTH|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+
|Manuel d'Héliogravure Et de Photogravure En Relief (Éd.1890) (Savoirs Et Traditions)                                                                                                        |84          |
|"DCGARING Microfiber Throw Blanket Warm Fuzzy Plush Fleece Blanket Twin Size Mandala Design Meditation Hippie Style Lightweight Warm Luxury Blanket Super Soft for Bed/Couch/Sofa 40""x

In [0]:
df = df.withColumnRenamed("PRODUCT_TYPE_ID", "CATEGORY_ID")
df.printSchema()

root
 |-- PRODUCT_ID: integer (nullable = true)
 |-- TITLE: string (nullable = true)
 |-- BULLET_POINTS: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- CATEGORY_ID: integer (nullable = true)
 |-- _corrupt_record: string (nullable = true)
 |-- Data_Source: string (nullable = false)
 |-- TITLE_LENGTH: integer (nullable = true)



In [0]:
df = df.withColumn(
    "PRODUCT_ID",
    col("PRODUCT_ID").cast("long")
)
df.printSchema()

root
 |-- PRODUCT_ID: long (nullable = true)
 |-- TITLE: string (nullable = true)
 |-- BULLET_POINTS: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- CATEGORY_ID: integer (nullable = true)
 |-- _corrupt_record: string (nullable = true)
 |-- Data_Source: string (nullable = false)
 |-- TITLE_LENGTH: integer (nullable = true)



In [0]:
from pyspark.sql.functions import col
for c in df.columns:
    print(c, ":", df.filter(col(c).isNull()).count())

PRODUCT_ID : 0
TITLE : 5
BULLET_POINTS : 274634
DESCRIPTION : 0
PRODUCT_TYPE_ID : 5065


In [0]:
df = df.fillna({
    "TITLE": "Unknown Product"
})

In [0]:
df = df.fillna({
    "BULLET_POINTS": "No Bullet Points"
})

In [0]:
df = df.fillna({
    "DESCRIPTION": "No Description"
})

In [0]:
df = df.dropna(subset=["PRODUCT_ID"])

In [0]:
df = df.dropDuplicates()

In [0]:
df = df.fillna({
    "TITLE": "Unknown Product",
    "BULLET_POINTS": "No Bullet Points"
})

In [0]:
df = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .option("inferSchema", "true") \
    .csv("/Volumes/workspace/default/amazon_data/test.csv")

In [0]:
print(df.columns)

['PRODUCT_ID', 'TITLE', 'BULLET_POINTS', 'DESCRIPTION', 'PRODUCT_TYPE_ID']


In [0]:
df = df.fillna({
    "DESCRIPTION": "No Description"
})

In [0]:
df.select("DESCRIPTION").show()

+--------------------+
|         DESCRIPTION|
+--------------------+
|      No Description|
|  all season choose.|
|Replacement for T...|
|Each pin comes wi...|
|      No Description|
|Description: Mate...|
|      No Description|
|      No Description|
|      No Description|
|      No Description|
|      No Description|
|Men’s Pure Cotton...|
|      No Description|
|Our 52'' wide x 7...|
|      No Description|
|Give a new style ...|
|      No Description|
|      No Description|
|      No Description|
|      No Description|
+--------------------+
only showing top 20 rows


In [0]:
print(df.columns)
df.printSchema()

['PRODUCT_ID', 'TITLE', 'BULLET_POINTS', 'DESCRIPTION', 'PRODUCT_TYPE_ID']
root
 |-- PRODUCT_ID: integer (nullable = true)
 |-- TITLE: string (nullable = true)
 |-- BULLET_POINTS: string (nullable = true)
 |-- DESCRIPTION: string (nullable = false)
 |-- PRODUCT_TYPE_ID: string (nullable = true)



In [0]:
df.write \
    .mode("overwrite") \
    .parquet("/Volumes/workspace/default/amazon_data/amazon_cleaned_test.csv")